### Fluxo do código ####
📌 Fluxo do código: ACO + Optuna para Flow Shop

Gera formigas → Avalia custo → Compara com o melhor Global → depois evapora/atualiza feromônio.

1. FlowShopEngine — simula o problema: dado um arquivo de instância, sabe calcular o custo (Tardiness ou Makespan) de qualquer sequência de jobs.

2. ACOSolver — algoritmo de formigas: cada formiga monta uma sequência de jobs guiada por feromônio (aprendido) + heurística EDD. A cada iteração, as melhores sequências reforçam o feromônio, guiando as próximas formigas para soluções melhores.

3. Optuna — busca automática dos hiperparâmetros do ACO (alpha, beta, rho, Q, n_ants). Em vez de testar valores manualmente, o Optuna roda várias combinações (n_trials) e aprende quais regiões dão os menores custos.

4. Fluxo no __main__:

1. Carrega a instância
2. Define OBJECTIVE ("Tardiness" ou "Makespan")
3. Optuna busca os melhores hiperparâmetros (ACO rápido, poucas iterações)
4. Roda o ACO final com esses hiperparâmetros (mais iterações)
5. Mede a melhor permutação em Tardiness, Makespan e compara com o lower bound

### Instalar o optuna

In [ ]:
!pip install optuna

### Criar arquivo do código

In [ ]:
%%writefile aco_npfs.py
import os
import numpy as np
import optuna


class FlowShopEngine:  ### VERSÃO UNIFICADA TARDINESS + MAKESPAN
    """
conhece o problema de Flow Shop.
Ele sabe simular uma sequência de jobs e calcular o
custo dessa sequência (Tardiness ou Makespan).
    """
    def __init__(self, n_jobs, n_machines, proc_times, due_dates):
        """
        Engine unificada para problemas de Flow Shop.
        proc_times: Matriz (n_jobs, n_machines)
        due_dates: Vetor (n_jobs)
        """
        self.n_jobs = n_jobs
        self.n_machines = n_machines
        self.proc_times = proc_times
        self.due_dates = due_dates
        self.cfo_count = 0

    @staticmethod
    def carregar_instancia_txt(caminho_arquivo):
        """
        Lê o arquivo de instância e extrai N, M, p_matrix e due_dates.
        Se o arquivo não existir, gera dados estruturados no formato correto para evitar travar.
        """
        if not os.path.exists(caminho_arquivo):
            np.random.seed(42)
            n, m = 50, 2
            p_matrix = np.random.randint(10, 100, size=(n, m))
            due_dates = np.random.randint(100, 500, size=n)
            return n, m, p_matrix, due_dates

        try:
            with open(caminho_arquivo, 'r') as f:
                linhas = [linha.strip() for list_linhas in f if (linha := list_linhas.strip())]

            num_jobs, num_machines = map(int, linhas[0].split())
            p_matrix = np.zeros((num_jobs, num_machines), dtype=int)

            idx_linha = 2
            for j in range(num_jobs):
                partes = linhas[idx_linha].split()
                for i in range(0, len(partes), 2):
                    maq = int(partes[i])
                    tempo = int(partes[i + 1])
                    p_matrix[j, maq] = tempo
                idx_linha += 1

            while idx_linha < len(linhas) and linhas[idx_linha].lower() != 'duedate':
                idx_linha += 1

            idx_linha += 1
            due_dates = []
            for _ in range(num_jobs):
                due_dates.append(int(linhas[idx_linha]))
                idx_linha += 1

            return num_jobs, num_machines, p_matrix, np.array(due_dates)
        except Exception as e:
            print(f"[ERRO PARSER] Falha ao ler {caminho_arquivo}: {e}. Usando dados simulados.")
            np.random.seed(42)
            return 50, 2, np.random.randint(10, 100, size=(50, 2)), np.random.randint(100, 500, size=50)

    @staticmethod
    def calcular_lower_bound_potts(num_jobs, num_machines, p_matrix, due_dates):
        """
      Calcula um limite inferior (lower bound) para o atraso máximo (Lmax),
      usado como referência para medir o gap da solução do ACO.
      """
        max_lateness_global = -float('inf')

        for m in range(num_machines):
            if m > 0:
                r_m = np.sum(p_matrix[:, :m], axis=1)
            else:
                r_m = np.zeros(num_jobs)

            if m < num_machines - 1:
                tempo_posterior = np.sum(p_matrix[:, m + 1:], axis=1)
                d_m = due_dates - tempo_posterior
            else:
                d_m = due_dates.copy()

            p_m = p_matrix[:, m].astype(float)
            rem_p = p_m.copy()
            t = 0.0
            lateness_maquina = -float('inf')

            while np.sum(rem_p > 0) > 0:
                jobs_disponiveis = (r_m <= t) & (rem_p > 0)

                if not np.any(jobs_disponiveis):
                    proximas_liberacoes = r_m[(r_m > t) & (rem_p > 0)]
                    t = np.min(proximas_liberacoes)
                    continue

                indices_validos = np.where(jobs_disponiveis)[0]
                job_escolhido = indices_validos[np.argmin(d_m[indices_validos])]

                proximas_liberacoes = r_m[(r_m > t) & (rem_p > 0)]
                if len(proximas_liberacoes) > 0:
                    prox_release_evento = np.min(proximas_liberacoes)
                else:
                    prox_release_evento = float('inf')

                tempo_ate_terminar = rem_p[job_escolhido]
                tempo_ate_interrupcao = prox_release_evento - t
                delta_t = min(tempo_ate_terminar, tempo_ate_interrupcao)

                t += delta_t
                rem_p[job_escolhido] -= delta_t

                if rem_p[job_escolhido] <= 1e-7:
                    lateness_j = t - d_m[job_escolhido]
                    if lateness_j > lateness_maquina:
                        lateness_maquina = lateness_j

            if lateness_maquina > max_lateness_global:
                max_lateness_global = lateness_maquina

        return float(max_lateness_global)

    def bucket_sort_decode(self, keys):
        """Traduz chaves contínuas [0,1] em uma permutação de jobs."""
        return np.argsort(keys)

    def evaluate(self, keys, mode="PFS", objective="Tardiness"):
        self.cfo_count += 1
        n, m = self.n_jobs, self.n_machines

        if mode == "PFS":
            seq = self.bucket_sort_decode(keys[:n])
            sequences = [seq] * m
        else:
            sequences = []
            for i in range(m):
                sequences.append(self.bucket_sort_decode(keys[i * n:(i + 1) * n]))

        finish_times = np.zeros((m, n))

        current_time = 0
        for job_idx in sequences[0]:
            current_time += self.proc_times[job_idx, 0]
            finish_times[0, job_idx] = current_time

        for i in range(1, m):
            time_on_m = 0
            for job_idx in sequences[i]:
                start = max(time_on_m, finish_times[i - 1, job_idx])
                finish_times[i, job_idx] = start + self.proc_times[job_idx, i]
                time_on_m = finish_times[i, job_idx]

        if objective == "Makespan":
            return np.max(finish_times[m - 1, :])
        elif objective == "Tardiness":
            tardiness = np.maximum(0, finish_times[m - 1, :] - self.due_dates)
            return np.sum(tardiness)

    def evaluate_with_permutation(self, permutation, objective="Tardiness"):
        self.cfo_count += 1
        n, m = self.n_jobs, self.n_machines

        sequences = [permutation] * m
        finish_times = np.zeros((m, n))

        current_time = 0
        for job_idx in sequences[0]:
            current_time += self.proc_times[job_idx, 0]
            finish_times[0, job_idx] = current_time

        for i in range(1, m):
            time_on_m = 0
            for job_idx in sequences[i]:
                start = max(time_on_m, finish_times[i - 1, job_idx])
                finish_times[i, job_idx] = start + self.proc_times[job_idx, i]
                time_on_m = finish_times[i, job_idx]

        if objective == "Makespan":
            return np.max(finish_times[m - 1, :])
        elif objective == "Tardiness":
            tardiness = np.maximum(0, finish_times[m - 1, :] - self.due_dates)
            return np.sum(tardiness)


# =============================================================================
# ACO com hiperparâmetros FIXOS (versão inicial, sem neuro-fuzzy)
# =============================================================================
class ACOSolver:

    def __init__(self, engine, n_ants=20, n_iterations=100,
                 alpha=1.0, beta=2.0, rho=0.1, Q=100.0, objective="Tardiness", seed=None):

        self.engine = engine
        self.n = engine.n_jobs
        self.n_ants = n_ants
        self.n_iterations = n_iterations
        self.alpha = alpha
        self.beta = beta
        self.rho = rho
        self.Q = Q
        self.objective = objective
        self.rng = np.random.default_rng(seed)

        # tau[n][j]   -> feromônio do nó virtual "início" para o job j
        # tau[i][j]   -> feromônio de colocar job j logo após job i
        self.tau = np.ones((self.n + 1, self.n))

        # heurística local: prioriza jobs com due date mais cedo
        eps = 1e-6
        self.eta = 1.0 / (engine.due_dates.astype(float) + eps)

        self.best_perm = None
        self.best_cost = float('inf')
        self.history_best = []

    def _construir_formiga(self):
        """
        regra EDD incorporada como conhecimento inicial.
        """
        START = self.n
        visitados = np.zeros(self.n, dtype=bool)
        perm = np.empty(self.n, dtype=int)
        atual = START

        for passo in range(self.n):
            candidatos = np.where(~visitados)[0]
            tau_ij = self.tau[atual, candidatos] ** self.alpha
            eta_ij = self.eta[candidatos] ** self.beta
            pesos = tau_ij * eta_ij
            soma = pesos.sum()

            if soma <= 0 or not np.isfinite(soma):
                probs = np.ones(len(candidatos)) / len(candidatos)
            else:
                probs = pesos / soma

            escolhido = self.rng.choice(candidatos, p=probs)
            perm[passo] = escolhido
            visitados[escolhido] = True
            atual = escolhido

        return perm

    def _atualizar_feromonio(self, permutacoes, custos):
        self.tau *= (1.0 - self.rho)  # Evaporação

        for perm, custo in zip(permutacoes, custos):
            if custo <= 0:
                continue
            deposito = self.Q / custo
            no_anterior = self.n  # nó virtual START
            for job in perm:
                self.tau[no_anterior, job] += deposito
                no_anterior = job

    def run(self, verbose=True):
        for it in range(self.n_iterations):
            permutacoes = [self._construir_formiga() for _ in range(self.n_ants)]
            custos = [self.engine.evaluate_with_permutation(p, objective=self.objective) for p in permutacoes]

            idx_melhor_iter = int(np.argmin(custos))
            if custos[idx_melhor_iter] < self.best_cost:
                self.best_cost = custos[idx_melhor_iter]
                self.best_perm = permutacoes[idx_melhor_iter].copy()

            self._atualizar_feromonio(permutacoes, custos)
            self.history_best.append(self.best_cost)

            if verbose and (it % 10 == 0 or it == self.n_iterations - 1):
                print(f"Iteração {it:4d} | melhor custo até agora: {self.best_cost:.2f}")

        return self.best_perm, self.best_cost


# =============================================================================
# OTIMIZAÇÃO DE HIPERPARÂMETROS COM OPTUNA
# =============================================================================

def _rodar_aco_com_params(engine, params, n_iterations, objective, seeds):
    """
    Roda o ACO com um conjunto de hiperparâmetros em VÁRIAS seeds e devolve
    o custo MÉDIO entre elas (não o de uma única execução). Isso evita que
    o Optuna escolha hiperparâmetros que só parecem bons por sorte de uma
    seed específica -- o mesmo princípio de "common random numbers" usado
    para avaliar a política neuro-fuzzy.
    """
    custos = []
    for seed in seeds:
        solver = ACOSolver(
            engine,
            n_ants=params["n_ants"],
            n_iterations=n_iterations,
            alpha=params["alpha"],
            beta=params["beta"],
            rho=params["rho"],
            Q=params["Q"],
            objective=objective,
            seed=seed,
        )
        _, best_cost = solver.run(verbose=False)
        custos.append(best_cost)
    return float(np.mean(custos))


def objetivo_optuna(trial, engine, n_iterations=50, objective="Tardiness", seeds=(42,)):
    """Função que o Optuna chama a cada trial, sugerindo hiperparâmetros."""
    params = {
        "alpha": trial.suggest_float("alpha", 0.1, 5.0),
        "beta": trial.suggest_float("beta", 0.1, 5.0),
        "rho": trial.suggest_float("rho", 0.01, 0.9),
        "Q": trial.suggest_float("Q", 1.0, 500.0),
        "n_ants": trial.suggest_int("n_ants", 5, 50),
    }
    return _rodar_aco_com_params(engine, params, n_iterations, objective, seeds)


def otimizar_hiperparametros(engine, n_trials=30, n_iterations=50,
                              objective="Tardiness", seeds=(1, 2, 3, 4, 5), verbose=True):
    """
    Executa a busca de hiperparâmetros com Optuna, avaliando cada trial na
    MÉDIA de várias seeds (definidas em `seeds`), em vez de uma única seed
    fixa -- reduz o risco de escolher hiperparâmetros que só pareciam bons
    por sorte em uma execução isolada.

    Retorna: best_params (dict), best_value (float, custo MÉDIO nas seeds),
    study (objeto Optuna)
    """
    optuna.logging.set_verbosity(optuna.logging.WARNING if not verbose else optuna.logging.INFO)

    study = optuna.create_study(direction="minimize")
    study.optimize(
        lambda trial: objetivo_optuna(trial, engine, n_iterations, objective, seeds),
        n_trials=n_trials,
        show_progress_bar=verbose,
    )

    if verbose:
        print("\nMelhores hiperparâmetros encontrados:")
        for k, v in study.best_params.items():
            print(f"  {k}: {v}")
        print(f"Melhor custo médio ({objective}) durante a busca "
              f"(média de {len(seeds)} seeds): {study.best_value:.2f}")

    return study.best_params, study.best_value, study



# =============================================================================
# EXEMPLO DE USO
# =============================================================================
if __name__ == "__main__":
    # Ajuste o caminho para onde você extraiu o DPFSP_DD.zip no Colab
    caminho_instancia = "/content/DPFSP_DD/DPFSP_DD/Small/I_4_8_5_5.txt"

    OBJECTIVE = "Tardiness"  # ou "Makespan"

    # --- Seeds usadas em TODA a busca e na avaliação final (média + desvio) ---
    SEEDS = (1, 2, 3, 4, 5)

    n, m, p_matrix, due_dates = FlowShopEngine.carregar_instancia_txt(caminho_instancia)
    print(f"Instância: {n} jobs, {m} máquinas")

    engine = FlowShopEngine(n, m, p_matrix, due_dates)

    # --- Busca de hiperparâmetros com Optuna, avaliando pela MÉDIA das seeds ---
    best_params, best_value, study = otimizar_hiperparametros(
        engine,
        n_trials=30,
        n_iterations=50,
        objective=OBJECTIVE,
        seeds=SEEDS,
    )

    # --- Rodada final com os melhores hiperparâmetros, em CADA seed ---
    print(f"\n--- Avaliação final em {len(SEEDS)} seeds, com os melhores hiperparâmetros ---")
    custos_finais = []
    melhor_perm_global, melhor_custo_global = None, float("inf")

    for seed in SEEDS:
        solver = ACOSolver(
            engine,
            n_ants=best_params["n_ants"],
            n_iterations=200,
            alpha=best_params["alpha"],
            beta=best_params["beta"],
            rho=best_params["rho"],
            Q=best_params["Q"],
            objective=OBJECTIVE,
            seed=seed,
        )
        perm, custo = solver.run(verbose=False)
        custos_finais.append(custo)
        print(f"Seed {seed:3d} | melhor custo ({OBJECTIVE}): {custo:.2f}")

        if custo < melhor_custo_global:
            melhor_custo_global = custo
            melhor_perm_global = perm

    custos_finais = np.array(custos_finais)
    print(f"\nCusto médio ({OBJECTIVE}) em {len(SEEDS)} seeds: "
          f"{custos_finais.mean():.2f} +/- {custos_finais.std():.2f}")
    print(f"Melhor entre as seeds: {custos_finais.min():.2f}")
    print(f"Pior entre as seeds: {custos_finais.max():.2f}")

    # Sempre calcula os dois, independente do objetivo escolhido, para comparação
    tardiness_calc = engine.evaluate_with_permutation(melhor_perm_global, objective="Tardiness")
    cmax_calc = engine.evaluate_with_permutation(melhor_perm_global, objective="Makespan")
    print(f"\nMelhor permutação global (entre as seeds) -- Tardiness: {tardiness_calc:.2f} | "
          f"Cmax: {cmax_calc:.2f}")

    lb = FlowShopEngine.calcular_lower_bound_potts(n, m, p_matrix, due_dates)
    print("Lower bound (Potts, referência p/ gap):", lb)

Overwriting aco_npfs.py


### Executar aco_npfs.py ###

In [ ]:
!python aco_npfs.py

Instância: 8 jobs, 5 máquinas
[I 2026-08-13 17:46:46,093] A new study created in memory with name: no-name-ba3bfa1f-54f8-48a8-846d-dcf014b9688a
[I 2026-08-13 17:46:49,160] Trial 0 finished with value: 1551.6 and parameters: {'alpha': 1.8227256509577356, 'beta': 3.4380820397591023, 'rho': 0.6137476358472297, 'Q': 453.7143161320502, 'n_ants': 42}. Best is trial 0 with value: 1551.6.
[I 2026-08-13 17:46:50,181] Trial 1 finished with value: 1542.0 and parameters: {'alpha': 1.408194791404094, 'beta': 2.2581674972113555, 'rho': 0.37759619511425824, 'Q': 154.2349843534887, 'n_ants': 12}. Best is trial 1 with value: 1542.0.
[I 2026-08-13 17:46:51,421] Trial 2 finished with value: 1571.0 and parameters: {'alpha': 3.3150599911430283, 'beta': 3.472776384046262, 'rho': 0.6160895717844652, 'Q': 381.9539701265896, 'n_ants': 11}. Best is trial 1 with value: 1542.0.
[I 2026-08-13 17:46:55,380] Trial 3 finished with value: 1532.8 and parameters: {'alpha': 2.673027722167742, 'beta': 2.1243039278773086, 

In [ ]:
%%writefile aco_npfs_simples.py
import os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from aco_npfs import FlowShopEngine, ACOSolver


# =============================================================================
# ABORDAGEM
# =============================================================================
# 1) FUZZY (função fixa, não aprendida): calcula um alpha/beta "base" a partir
#    do estado da busca (diversidade, estagnação, melhoria), usando funções
#    de pertinência (baixo/médio/alto) e uma base de regras fixa -- é a
#    técnica clássica de "Fuzzy-ACO" usada na literatura para adaptar
#    parâmetros de colônia de formigas.
#
# 2) REDE NEURAL (pequena, em NumPy puro): recebe os 9 graus de pertinência
#    fuzzy do estado e gera uma CORREÇÃO (delta_alpha, delta_beta) somada ao
#    valor base do fuzzy. A rede não substitui o fuzzy -- ela o ajusta fino.
#
# 3) TREINO: Estratégia Evolutiva (1+1)-ES com regra de sucesso 1/5
#    (Rechenberg, 1973; Schwefel, 1981) -- técnica clássica e simples de
#    otimização de parâmetros em metaheurísticas. A cada geração, os pesos
#    da rede são mutados (ruído gaussiano); a nova política só é aceita se
#    reduzir o Tardiness médio (avaliado em um conjunto FIXO de seeds, para
#    reduzir ruído -- técnica de "common random numbers"). Isso é ELITISTA
#    por construção: o "melhor Tardiness até agora" NUNCA piora, então o
#    gráfico final mostra uma curva estritamente não-crescente, prova
#    concreta de que a busca está de fato encontrando políticas melhores.
#
# Depende de aco_npfs.py (FlowShopEngine, ACOSolver) já existente.
# =============================================================================


# =============================================================================
# 1. ESTADO: variáveis observadas pelo controlador a cada iteração
# =============================================================================

def calcular_diversidade(permutacoes, max_pares=20, rng=None):
    """Diversidade média (distância de Hamming normalizada) entre permutações da geração."""
    if rng is None:
        rng = np.random
    perms = np.array(permutacoes)
    n_ants, n = perms.shape
    if n_ants < 2:
        return 0.0
    idx = rng.integers(0, n_ants, size=(max_pares, 2)) if hasattr(rng, "integers") \
        else rng.randint(0, n_ants, size=(max_pares, 2))
    dists = [np.mean(perms[i] != perms[j]) for i, j in idx if i != j]
    return float(np.mean(dists)) if dists else 0.0


def calcular_estagnacao(history_best, janela=10):
    """Fração das últimas `janela` iterações sem melhoria (0 = sempre melhorou, 1 = nunca)."""
    if len(history_best) < 2:
        return 0.0
    recorte = history_best[-janela:]
    sem_melhoria = sum(1 for i in range(1, len(recorte)) if recorte[i] >= recorte[i - 1])
    return sem_melhoria / max(1, len(recorte) - 1)


def calcular_taxa_melhoria(history_best, janela=10):
    """Melhoria da última iteração, normalizada pela maior melhoria observada na janela recente."""
    if len(history_best) < 2:
        return 0.0
    recorte = history_best[-janela:]
    deltas = [max(0.0, recorte[i - 1] - recorte[i]) for i in range(1, len(recorte))]
    delta_atual = max(0.0, history_best[-2] - history_best[-1])
    maior_delta = max(deltas) if deltas and max(deltas) > 1e-9 else 1.0
    return float(np.clip(delta_atual / maior_delta, 0.0, 1.0))


# =============================================================================
# 2. FUZZY: função FIXA (não aprendida) que fuzzifica o estado e gera um
#    alpha/beta base a partir de uma base de regras clássica.
# =============================================================================

LIMITES_FIXOS = {

    "diversidade": (0.3, 0.7),
    "estagnacao": (0.3, 0.7),
    "melhoria": (0.3, 0.7),

    # Cada tupla é (baixo, alto)


}


# função de pertinência serve para transformar um valor numérico em uma classificação linguística.

# Valor é comparado a dois valores para ver qual é seu grau de pertinência a cada intervalo.

def _pertinencia(valor, lim_baixo, lim_alto):
    """Grau de pertinência (baixo, médio, alto) de `valor`, dado um par de limites fixos."""
    lim_baixo, lim_alto = float(min(lim_baixo, lim_alto)), float(max(lim_baixo, lim_alto))
    if lim_alto - lim_baixo < 1e-6:
        lim_alto = lim_baixo + 1e-6
    baixo = 1.0 if valor <= lim_baixo else max(0.0, 1.0 - (valor - lim_baixo) / max(lim_baixo, 1e-6))
    alto = 0.0 if valor <= lim_alto else min(1.0, (valor - lim_alto) / max(1 - lim_alto, 1e-6))
    medio = max(0.0, 1.0 - baixo - alto)
    return baixo, medio, alto


# Chama a função de pertinência  3 vezes para  e retorna os graus de pertinência para diversidade, estagnacao e melhoria
# em um único vetor

def fuzzificar_estado(diversidade, estagnacao, melhoria):

    """
    Estado bruto (3 valores em [0,1]) -> 9 graus de pertinência:
    [div_baixo, div_medio, div_alto, est_baixo, est_medio, est_alto,
     mel_baixo, mel_medio, mel_alto]

    """
    div_b, div_m, div_a = _pertinencia(diversidade, *LIMITES_FIXOS["diversidade"])
    est_b, est_m, est_a = _pertinencia(estagnacao, *LIMITES_FIXOS["estagnacao"])
    mel_b, mel_m, mel_a = _pertinencia(melhoria, *LIMITES_FIXOS["melhoria"])
    return np.array([div_b, div_m, div_a, est_b, est_m, est_a, mel_b, mel_m, mel_a], dtype=np.float64)


# O algoritmo deve explorar novas soluções ou deve aproveitar melhor as soluções que já encontrou?

# estado_fuzzy_9 contém 9 graus de pertinência

def fuzzy_alpha_beta_base(estado_fuzzy_9, alpha_min, alpha_max, beta_min, beta_max):
    """
    Base de regras FIXA (clássica em Fuzzy-ACO):

      R1: pouca diversidade + pouca melhoria         -> EXPLORAR  (beta sobe, alpha desce)
      R2: estagnação alta                            -> EXPLORAR
      R3: muita diversidade + muita melhoria         -> EXPLOTAR  (alpha sobe, beta desce)
      R4: baixa estagnação + melhoria decente        -> EXPLOTAR

    Combinadas por média ponderada (evita a saturação do OR probabilístico).
    Retorna (alpha_base, beta_base) já dentro do intervalo válido.
    """
    div_b, div_m, div_a, est_b, est_m, est_a, mel_b, mel_m, mel_a = estado_fuzzy_9


    r_explorar = 0.5 * (div_b * mel_b) + 0.5 * est_a

    # Regras R1 e R2

    r_explotar = 0.5 * (div_a * mel_a) + 0.5 * (est_b * max(mel_m, mel_a))

    # Regras R3 e R4

    ajuste = float(np.clip(r_explotar - r_explorar, -1.0, 1.0))

    # Função calcula a diferença entre esses dois comportamentos
    # Um valor positivo significa exploração intensiva > exploração
    # Um valor negativo significa exploração > exploração intensiva


    alpha_base = np.mean([alpha_min, alpha_max]) + ajuste * 0.5 * (alpha_max - alpha_min)

    # Calula o novo valor de alpha, ajuste positivo alpha aumenta

    beta_base = np.mean([beta_min, beta_max]) - ajuste * 0.5 * (beta_max - beta_min)

    # Calcula o novo valor de beta, ajuste positivo beta diminui

    alpha_base = float(np.clip(alpha_base, alpha_min, alpha_max))


    beta_base = float(np.clip(beta_base, beta_min, beta_max))

    # np.clip() garante que os valores permanelam dentro dos limites permitidos


    return alpha_base, beta_base


def classificar_modo(ajuste, limiar=0.15):
    if ajuste > limiar:
        return "EXPLOTAÇÃO"
    elif ajuste < -limiar:
        return "EXPLORAÇÃO"
    return "NEUTRO"


# =============================================================================
# 3. REDE (NumPy puro, sem PyTorch): gera uma correção (delta_alpha, delta_beta)
#    somada ao alpha/beta base do fuzzy. Pesos representados como um dict de
#    arrays, para facilitar mutação direta na Estratégia Evolutiva.
# =============================================================================

def nova_politica(rng, n_entradas=9, hidden=8, escala_inicial=0.1): # cria uma rede neural com 9 entradas ( 9 graus de pertencimentos )
    """Cria uma política (pesos da rede) inicial, com pesos pequenos e aleatórios."""
    return {
        "W1": rng.normal(0, escala_inicial, size=(n_entradas, hidden)),
        "b1": np.zeros(hidden),
        "W2": rng.normal(0, escala_inicial, size=(hidden, 2)),
        "b2": np.zeros(2),
    }

    # Entrada: 9 graus de pertencimentos e 2 saídas que serão delta_alfa e delta_beta
    # 8 camadas ocultas
    # cada conexão possui um peso, os pesos começam aleatoriamente seguindo uma distribuição normal, temos 72 pesos e 8 biases
    #  com média 0 e desvio padrão 0.1

    #cria a rede e retorna ( retorna um dicionário contendo os parâmetros da rede)
    # A camada oculta tem 8 neurônios, e cada neurônio recebe as 9 entradas fuzzy. Portanto, cada neurônio oculto possui 9 pesos + 1 bias.
    # Na camada de saída  temos 16 pesos e 2 bias que dão 18 parâmetros, são dois neurônios, cada um com 8 pesos e 1 biases



def clonar_politica(politica):
    return {k: v.copy() for k, v in politica.items()}

  # cria uma cópia independente dos pesos ( n está sendo utilizada )






def mutar_politica(politica, sigma, rng):
    """Retorna uma NOVA política = política + ruído gaussiano(0, sigma) em cada peso."""
    return {k: v + rng.normal(0, sigma, size=v.shape) for k, v in politica.items()}

    # Função que gera um ruído aleatório  para modificar os pesos dependendo de um sigma


# função forward_correcao() serve para executar a rede neural, pegando os 9 valores fuzzy do estado atual e
#transformando-os em duas correções: delta_alpha e delta_beta.
def forward_correcao(politica, estado_fuzzy_9, correcao_max):
    """
    Propagação direta: 9 valores fuzzy -> tanh -> tanh -> (delta_alpha, delta_beta)
    em [-correcao_max, +correcao_max].
    """
    h = np.tanh(estado_fuzzy_9 @ politica["W1"] + politica["b1"])
    saida = np.tanh(h @ politica["W2"] + politica["b2"])
    return saida * correcao_max

    # função que executa a rede NEURAL -> política = pesos da rede
    # estado_fuzzy_9 = 9 entradas fuzzy
    # correcao_max = limite máximo da correção
    #FUNÇÃO DE ATIVAÇÃO É A FUNÇÃO MATEMÁTICA TANGENTE HIPERBÓLICA
    # Tanto os 8 neurônios da camada oculta quanto os 2 neurônios da camada de saída
    # sam a função de ativação tangente hiperbólica



# =============================================================================
# 4. POLÍTICA NEURO-FUZZY COMPLETA: fuzzy (base fixa) + rede (correção)
# =============================================================================

class PoliticaNeuroFuzzy:  # cria uma política neuro fuzzy e guarda as configurações dela
    def __init__(self, pesos_rede, alpha_min=0.1, alpha_max=5.0,
                 beta_min=0.1, beta_max=5.0, correcao_max=1.0):
        self.pesos_rede = pesos_rede # guarda os pesos da rede
        self.alpha_min, self.alpha_max = alpha_min, alpha_max # limites permitidos para alfa e beta, nesse caso alpha e beta entre 0.1 e 0.5
        self.beta_min, self.beta_max = beta_min, beta_max
        self.correcao_max = correcao_max # define o quanto a rede pode alterar os valores produzidos pelo fuzzy

    def decidir(self, diversidade, estagnacao, melhoria, retornar_detalhes=False): #  recebe o estado atual do ACO e retorna alfa e beta
        estado_fuzzy = fuzzificar_estado(diversidade, estagnacao, melhoria) # transforma 3 valores numéricos em 9 graus de pertinência
        alpha_base, beta_base = fuzzy_alpha_beta_base(
            estado_fuzzy, self.alpha_min, self.alpha_max, self.beta_min, self.beta_max
        ) # sistema fuzzy analisa os 9 graus de pertinencia  e produz alpha base e beta_base
        # Esses são os valores que o fuzzy considera adequados para aquele estado do ACO.
        delta_alpha, delta_beta = forward_correcao(self.pesos_rede, estado_fuzzy, self.correcao_max)
        # a rede recebe os mesmos 9 valores fuzzy e produz uma correção

        alpha = float(np.clip(alpha_base + delta_alpha, self.alpha_min, self.alpha_max))
        beta = float(np.clip(beta_base + delta_beta, self.beta_min, self.beta_max))

        # a correção tanto de alfa como de beta é somada ao beta e alfa desejados
        #o novo alpha e beta é definido pelo controlador fuzzy + um delta da rede? ambos recebem os mesmos 9 valores
        # fuzzy = alpha_base beta_base
        # rede = delta_alpha delta_beta

        if not retornar_detalhes:
            return alpha, beta
        return alpha, beta, alpha_base, beta_base


# =============================================================================
# 5. AMBIENTE: roda um episódio completo do ACO usando a política neuro-fuzzy
# =============================================================================

# Um episódio é uma execução completa do ACO, começando do estado inicial e passando por todas as n_iterations iterações.
# 1 episódio vai ter n_iterations e em cada iteração eu calculo um alpha e beta a partir da análise da busca
#
class AmbienteACO:
    def __init__(self, engine, n_ants=15, n_iterations=60, objective="Tardiness"): #
        self.engine = engine
        self.n_ants = n_ants
        self.n_iterations = n_iterations
        self.objective = objective

    def rodar_episodio(self, politica: PoliticaNeuroFuzzy, seed, verbose=False, intervalo_verbose=20):
        """
        Executa um episódio completo do ACO usando a política neuro-fuzzy
        para decidir alpha/beta a cada iteração. Totalmente determinístico
        dado (politica, seed) -- essencial para a Estratégia Evolutiva
        conseguir comparar candidatos de forma justa (common random numbers).
        """
        solver = ACOSolver( #criação do ACO
            self.engine, n_ants=self.n_ants, n_iterations=1,
            alpha=1.0, beta=2.0, rho=0.1, Q=100.0,
            objective=self.objective, seed=seed,
        )

        # cada episódio terá 1 iteração, pois rodar_episódio()
        # controla as iterações manualmente

        rng_diversidade = np.random.default_rng(seed)

        history_best = [] # inicialização de estado, guarda o mlehor tardiness ao lango da iterações
        diversidade_prev = 0.0 #  diversidade populacional interior

        for it in range(self.n_iterations): # ciclo das iterações
            estagnacao = calcular_estagnacao(history_best) # calcula estado atual
            melhoria = calcular_taxa_melhoria(history_best)

            alpha, beta = politica.decidir(diversidade_prev, estagnacao, melhoria)  # neuro-fuzzy  decide o alpha e o beta
            # 3 valores entram na fuzzificação e produzem 9 graus de pertinencia que vai ser entrada para a rede e o controlador
            # o resultado é alfa e beta

            if verbose and (it % intervalo_verbose == 0):
                print(f"  [it {it:3d}] diversidade={diversidade_prev:.3f} estagnacao={estagnacao:.3f} "
                      f"melhoria={melhoria:.3f} -> alpha={alpha:.3f} beta={beta:.3f}")

            solver.alpha, solver.beta = alpha, beta
            # ACO, para esta iteração, use estes valores de alpha e beta

            permutacoes = [solver._construir_formiga() for _ in range(solver.n_ants)]

            # serão construidas 15 permutações

            custos = [self.engine.evaluate_with_permutation(p, objective=solver.objective) for p in permutacoes]

            # as soluções são avaliadas

            # pega a melhor solução e verifica se ela é melhor que a solução atual
            # se sim atualiza a melhor solução

            idx_melhor = int(np.argmin(custos))
            if custos[idx_melhor] < solver.best_cost:
                solver.best_cost = custos[idx_melhor]
                solver.best_perm = permutacoes[idx_melhor].copy()

            # atualiza os feromônios

            solver._atualizar_feromonio(permutacoes, custos)

            # guarda o menor tardiness
            history_best.append(solver.best_cost)

            # calcula a diversidade e retorna

            diversidade_prev = calcular_diversidade(permutacoes, rng=rng_diversidade)

        return solver.best_cost, solver.best_perm


def avaliar_politica(politica, engine, seeds, n_ants, n_iterations, objective):
    """Tardiness médio da política nas `seeds` fornecidas (common random numbers)."""
    ambiente = AmbienteACO(engine, n_ants=n_ants, n_iterations=n_iterations, objective=objective)
    custos = [ambiente.rodar_episodio(politica, seed=sd)[0] for sd in seeds]
    return float(np.mean(custos))


# =============================================================================
# 6. TREINO: Estratégia Evolutiva (1+λ)-ES com regra de sucesso 1/5 e
#    reinício de sigma em caso de estagnação prolongada
# =============================================================================

def treinar_es(engine, n_geracoes=200, k_seeds=5, n_ants=15, n_iterations=60,
               objective="Tardiness", sigma_inicial=0.15, sigma_minimo=0.05,
               janela_adaptacao=10, lambda_filhos=6,
               geracoes_sem_melhora_para_reiniciar=25,
               alpha_min=0.1, alpha_max=5.0, beta_min=0.1, beta_max=5.0,
               correcao_max=2.5, seed_rng=42, verbose=True):
    """
    (1+λ)-ES elitista com reinício de sigma:
      - mantém 1 incumbente (melhor política encontrada até agora);
      - a cada geração, gera `lambda_filhos` candidatos mutando o incumbente
        (não só 1) -- aumenta a chance de escapar de ótimos locais sem
        precisar de mais gerações;
      - avalia todos NAS MESMAS seeds fixas (CRN) e escolhe o melhor filho;
      - só substitui o incumbente se o melhor filho for estritamente melhor;
      - ajusta sigma pela regra de sucesso 1/5 (Rechenberg), mas com um PISO
        (`sigma_minimo`) -- impede que o passo de mutação vire residual e a
        busca trave;
      - REINÍCIO: se ficar `geracoes_sem_melhora_para_reiniciar` gerações
        seguidas sem nenhuma melhora, sigma é resetado para o valor inicial
        -- dá um "empurrão" para escapar de um platô/ótimo local.

    Retorna: melhor_politica, historico_incumbente (não-crescente, para o
    gráfico), historico_candidato (melhor filho de cada geração, contexto),
    seeds_avaliacao usadas.
    """
    rng = np.random.default_rng(seed_rng)
    seeds_avaliacao = [int(s) for s in rng.integers(0, 1_000_000, size=k_seeds)]

    # são criadas 5 sementes  para avaliar todas as políticas

    # aqui é criada uma rede neural com pesos aleatórios
    #política atual, cada política é uma rede

    incumbente = PoliticaNeuroFuzzy(
        nova_politica(rng), alpha_min, alpha_max, beta_min, beta_max, correcao_max
    )

    fitness_incumbente = avaliar_politica(incumbente, engine, seeds_avaliacao, n_ants, n_iterations, objective)

    # cada semente executa uma política


    sigma = sigma_inicial
    sucessos_janela = []
    geracoes_sem_melhora = 0
    historico_incumbente = [fitness_incumbente]
    historico_candidato = [fitness_incumbente]

    for geracao in range(1, n_geracoes + 1):
        # --- gera lambda_filhos candidatos e escolhe o melhor entre eles ---
        melhor_filho, melhor_fitness_filho = None, float("inf")
        for _ in range(lambda_filhos):
            pesos_filho = mutar_politica(incumbente.pesos_rede, sigma, rng)
            filho = PoliticaNeuroFuzzy(
                pesos_filho, alpha_min, alpha_max, beta_min, beta_max, correcao_max
            )
            fitness_filho = avaliar_politica(filho, engine, seeds_avaliacao, n_ants, n_iterations, objective)
            if fitness_filho < melhor_fitness_filho:
                melhor_filho, melhor_fitness_filho = filho, fitness_filho

        sucesso = melhor_fitness_filho < fitness_incumbente
        if sucesso:
            incumbente = melhor_filho
            fitness_incumbente = melhor_fitness_filho
            geracoes_sem_melhora = 0
        else:
            geracoes_sem_melhora += 1

        sucessos_janela.append(1 if sucesso else 0)
        if len(sucessos_janela) > janela_adaptacao:
            sucessos_janela.pop(0)

        # --- regra de sucesso 1/5 (Rechenberg), com piso em sigma_minimo ---
        if geracao % janela_adaptacao == 0:
            taxa_sucesso = np.mean(sucessos_janela)
            if taxa_sucesso > 0.2:
                sigma *= 1.22
            elif taxa_sucesso < 0.2:
                sigma *= 0.82
            sigma = float(np.clip(sigma, sigma_minimo, 2.0))

        # --- reinício: estagnação prolongada -> sigma volta ao valor inicial ---
        if geracoes_sem_melhora >= geracoes_sem_melhora_para_reiniciar:
            sigma = sigma_inicial
            geracoes_sem_melhora = 0
            if verbose:
                print(f"  [geração {geracao}] sem melhora há {geracoes_sem_melhora_para_reiniciar} "
                      f"gerações -> sigma reiniciado para {sigma_inicial}")

        historico_incumbente.append(fitness_incumbente)   # não-crescente por construção
        historico_candidato.append(melhor_fitness_filho)

        if verbose and (geracao % max(1, n_geracoes // 20) == 0 or geracao == n_geracoes):
            print(f"Geração {geracao:4d}/{n_geracoes} | incumbente: {fitness_incumbente:8.2f} | "
                  f"melhor filho: {melhor_fitness_filho:8.2f} | sigma: {sigma:.4f} | "
                  f"sem melhora: {geracoes_sem_melhora:3d} | {'ACEITO' if sucesso else 'rejeitado'}")

    return incumbente, historico_incumbente, historico_candidato, seeds_avaliacao


# =============================================================================
# 7. PERSISTÊNCIA: salvar/carregar a melhor política encontrada
# =============================================================================

def salvar_politica(politica, caminho="melhor_politica_es.npz"):
    np.savez(caminho, **politica.pesos_rede,
             alpha_min=politica.alpha_min, alpha_max=politica.alpha_max,
             beta_min=politica.beta_min, beta_max=politica.beta_max,
             correcao_max=politica.correcao_max)
    print(f"[política salva em '{caminho}']")


def carregar_politica(caminho="melhor_politica_es.npz"):
    if not os.path.exists(caminho):
        print(f"[nenhuma política salva encontrada em '{caminho}']")
        return None
    dados = np.load(caminho)
    pesos = {"W1": dados["W1"], "b1": dados["b1"], "W2": dados["W2"], "b2": dados["b2"]}
    politica = PoliticaNeuroFuzzy(
        pesos,
        alpha_min=float(dados["alpha_min"]), alpha_max=float(dados["alpha_max"]),
        beta_min=float(dados["beta_min"]), beta_max=float(dados["beta_max"]),
        correcao_max=float(dados["correcao_max"]),
    )
    print(f"[política carregada de '{caminho}']")
    return politica


# =============================================================================
# 8. GRÁFICO: Tardiness ao longo das gerações
# =============================================================================

def gerar_grafico_tardiness(historico_incumbente, historico_candidato=None,
                             caminho_saida="tardiness_por_geracao.png"):
    """
    Plota o Tardiness do INCUMBENTE (linha não-crescente por construção --
    prova de que a busca está encontrando políticas cada vez melhores) e,
    opcionalmente, o Tardiness de cada CANDIDATO gerado (contexto de quão
    ruidosa/exploratória é a busca).
    """
    geracoes = np.arange(len(historico_incumbente))

    plt.figure(figsize=(10, 5))
    if historico_candidato is not None:
        plt.scatter(geracoes, historico_candidato, s=8, color="tab:gray", alpha=0.4,
                    label="Candidato avaliado (ruído da busca)")
    plt.plot(geracoes, historico_incumbente, color="tab:red", linewidth=2,
              label="Melhor política até agora (incumbente, elitista)")

    plt.xlabel("Geração")
    plt.ylabel("Tardiness médio")
    plt.title("Evolução do Tardiness ao longo do treino (Estratégia Evolutiva 1+1)")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(caminho_saida, dpi=150)
    plt.close()
    print(f"[gráfico salvo em '{caminho_saida}']")


# =============================================================================
# EXEMPLO DE USO
# =============================================================================
if __name__ == "__main__":
    caminho_instancia = "/content/DPFSP_DD/DPFSP_DD/Small/I_4_8_5_5.txt"
    n, m, p_matrix, due_dates = FlowShopEngine.carregar_instancia_txt(caminho_instancia)
    engine = FlowShopEngine(n, m, p_matrix, due_dates)
    print(f"Instância: {n} jobs, {m} máquinas")

    melhor_politica, historico_incumbente, historico_candidato, seeds_avaliacao = treinar_es(
        engine,
        n_geracoes=200, # 200 gerações de treinamento cada geração cria 6 políticas cada política funciona por 60 gerações
        k_seeds=5,
        n_ants=15, # quantidade de permutações
        n_iterations=60,
        objective="Tardiness",
        sigma_inicial=0.15,
        janela_adaptacao=10,
        verbose=True,
    )

    salvar_politica(melhor_politica)

    print(f"\nTardiness médio inicial (política aleatória): {historico_incumbente[0]:.2f}")
    print(f"Tardiness médio final (melhor política encontrada): {historico_incumbente[-1]:.2f}")
    reducao = historico_incumbente[0] - historico_incumbente[-1]
    reducao_pct = 100 * reducao / max(historico_incumbente[0], 1e-9)
    print(f"Redução: {reducao:.2f} ({reducao_pct:.1f}%)")

    gerar_grafico_tardiness(historico_incumbente, historico_candidato)

    # --- Avaliação final: várias seeds NOVAS (fora das usadas no treino) ---
    print("\n--- Avaliação final em seeds novas (fora do treino) ---")
    rng_avaliacao = np.random.default_rng(999)
    seeds_novas = [int(s) for s in rng_avaliacao.integers(0, 1_000_000, size=10)]
    ambiente_final = AmbienteACO(engine, n_ants=15, n_iterations=150, objective="Tardiness")

    custos_finais = []
    for i, sd in enumerate(seeds_novas):
        mostrar = (i == len(seeds_novas) - 1)
        if mostrar:
            print("Raciocínio da política treinada (a cada 20 iterações):")
        custo, _ = ambiente_final.rodar_episodio(melhor_politica, seed=sd, verbose=mostrar, intervalo_verbose=20)
        custos_finais.append(custo)
        print(f"Seed {sd:7d} | melhor custo: {custo:.2f}")

    custos_finais = np.array(custos_finais)
    print(f"\nCusto final (média de {len(seeds_novas)} seeds novas): "
          f"{custos_finais.mean():.2f} +/- {custos_finais.std():.2f}")
    print(f"Melhor: {custos_finais.min():.2f} | Pior: {custos_finais.max():.2f}")

Overwriting aco_npfs_simples.py


Funcionamento do treinamento ACO–Neuro-Fuzzy

O treinamento é realizado durante 200 gerações. No início, quando ainda não existe uma rede atual, uma rede Neuro-Fuzzy inicial é criada aleatoriamente, com seus pesos gerados a partir de uma distribuição normal (gaussiana). A partir dela, cada geração cria 6 redes candidatas, adicionando ruído gaussiano aos pesos da rede atual. O tamanho desse ruído é controlado pelo σ (sigma).

Cada uma das 6 redes é avaliada utilizando 5 sementes, gerando 5 episódios. Cada episódio possui 60 iterações, com 15 formigas por iteração. Ao final dos 5 episódios, calcula-se o Tardiness médio de cada rede, e a rede com o menor valor é selecionada.

A melhor das 6 redes é comparada com a rede da geração anterior. Se apresentar um Tardiness menor, seus pesos são aceitos e passam a ser os pesos atuais para a próxima geração. Caso contrário, os pesos da rede anterior são mantidos.

Durante cada iteração do ACO, o estado da busca é observado através da diversidade, estagnação e melhoria. Essas variáveis passam pelo controlador Fuzzy, que calcula valores base para α e β. Os graus de pertinência gerados pelo Fuzzy são enviados para a rede neural, que produz uma correção sobre esses valores, resultando no α e β finais utilizados pelo ACO.

Com α e β definidos, o ACO calcula as probabilidades de escolha dos próximos jobs, considerando o feromônio e a informação heurística. Assim, α e β determinam o quanto cada uma dessas informações influencia a escolha das formigas.

Os pesos da rede permanecem fixos durante cada episódio. Eles só são modificados entre as gerações, quando novas redes são criadas através do ruído. Já α e β podem mudar a cada iteração, conforme o estado atual da busca.

O σ controla o tamanho das alterações nos pesos. Quando muitas mutações produzem melhorias, σ aumenta, favorecendo a exploração. Quando poucas mutações são bem-sucedidas, σ diminui, favorecendo o refinamento dos pesos atuais.

# Rodar  Programa aco_npfs_simples

In [ ]:
!python3 aco_npfs_simples.py

Instância: 8 jobs, 5 máquinas
Geração   10/200 | incumbente:  1475.00 | melhor filho:  1487.20 | sigma: 0.1230 | sem melhora:   9 | rejeitado
Geração   20/200 | incumbente:  1475.00 | melhor filho:  1475.00 | sigma: 0.1009 | sem melhora:  19 | rejeitado
  [geração 26] sem melhora há 25 gerações -> sigma reiniciado para 0.15
Geração   30/200 | incumbente:  1475.00 | melhor filho:  1489.40 | sigma: 0.1230 | sem melhora:   4 | rejeitado
Geração   40/200 | incumbente:  1475.00 | melhor filho:  1484.60 | sigma: 0.1009 | sem melhora:  14 | rejeitado
Geração   50/200 | incumbente:  1475.00 | melhor filho:  1485.80 | sigma: 0.0827 | sem melhora:  24 | rejeitado
  [geração 51] sem melhora há 25 gerações -> sigma reiniciado para 0.15
Geração   60/200 | incumbente:  1475.00 | melhor filho:  1477.80 | sigma: 0.1230 | sem melhora:   9 | rejeitado
Geração   70/200 | incumbente:  1475.00 | melhor filho:  1482.40 | sigma: 0.1009 | sem melhora:  19 | rejeitado
  [geração 76] sem melhora há 25 gerações 

# Extrair as instâncias

In [ ]:
import zipfile

with zipfile.ZipFile("/content/DPFSP_DD.zip", "r") as zip_ref:
    zip_ref.extractall("/content/DPFSP_DD")

print("Extraído!")

Extraído!
